# Algoritmi MCTS. Aplicație: Connect Four
 - Tudor Berariu
 - Andrei Olaru

## Scopul laboratorului

Scopul acestui laborator este acela de a implementa un algoritm din familia **MCTS** (_Monte Carlo Tree Search_), și anume **UCT** (_Upper Confidence Bound for Trees_).

Pentru a testa algoritmul vom folosi jocul _Connect Four_.

Prima parte a laboratorului construiește funcțiile necesare desfășurarea jocului _Connect Four_, iar cea de-a doua parte urmărește implementarea pas cu pas a algoritmului **UCT**.

#### Resurse

https://en.wikipedia.org/wiki/Monte_Carlo_tree_search

Slides curs IA despre Căutare în Jocuri

## Jocul _Connect Four_

### Descrierea jocului

Jocul _Connect Four_ [(link)](https://en.wikipedia.org/wiki/Connect_Four) lucrează cu o matrice verticală de **înălțime 6** și **lățime 7** în care doi jucători dau drumul unor jetoane de culori diferite (un jucător are jetoane roșii, iar celălalt albastre). La fiecare mutare, un jucător alege o coloană din cele 7 și dă drumul jetonului. Acesta _cade_, așezându-se pe prima poziție liberă din acea coloană. Într-o coloană nu se pot așeza mai mult de șase jetoane. Câștigă acel jucător care reușește să așeze *patru* dintre jetoanele lui (de aceeași culoare) într-o linie pe orizontală, verticală sau diagonală.

### Reprezentarea stărilor

Starea jocului va fi reprezentată printr-un tuplu format din două elemente:
 - o listă ce va conține 7 liste corespunzătoare celor 7 coloane
   + fiecare listă va avea lungimea 6 și va conține 0 (poziție liberă), 1 (jeton roșu) și 2 (jeton albastru)
   + poziția 0 din listă corespunde rândului cel mai de jos
 - indicatorul jucătorului ce trebuie să _mute_: 1 pentru roșu, 2 pentru jucătorul albastru.

In [1]:
"""Cu ce se ocupa Monte Carlo Tree Search (MCTS) si Upper Confidence Bound for Trees (UCT)?
Raspuns: Monte Carlo Tree Search (MCTS) este o metodă de căutare utilizată în inteligența artificială pentru a lua decizii în jocuri și alte probleme de luare a deciziilor.

MCTS combină explorarea și exploatarea pentru a construi un arbore de căutare bazat pe simulări ale jocului. 
Prin exploatare se intelege ca se aleg nodurile care au dat rezultate bune în trecut, 
                iar prin explorare se intelege ca se aleg nodurile care nu au fost explorate suficient pentru a descoperi potențialul lor.

Upper Confidence Bound for Trees (UCT) este o strategie specifică utilizată în MCTS pentru a selecta nodurile din arborele de căutare, echilibrând între explorarea nodurilor neexplorate 
și exploatarea nodurilor care au dat rezultate bune în trecut. UCT utilizează o formulă care ia în considerare atât valoarea estimată a nodului, cât și numărul de vizite pentru a decide 
care nod să fie explorat în continuare.
"""

# Dimensiunile matricei
HEIGHT, WIDTH = 6, 7

# Pozițiile din tuplul ce constituie o stare
BOARD, NEXT_PLAYER = 0, 1

# Jucătorii
RED, BLUE, DRAW = 1, 2, 3
name = ["", "ROȘU", "ALBASTRU", "REMIZĂ"]

# Funcție ce întoarce o stare inițială
def init_state():
    return ([[0 for row in range(HEIGHT)] for col in range(WIDTH)], RED)
"""
Ce inseamna return ([[0 for row in range(HEIGHT)] for col in range(WIDTH)], RED) ?
Această linie de cod returnează o stare inițială pentru jocul de Connect Four.
- `[[0 for row in range(HEIGHT)] for col in range(WIDTH)]` creează o matrice de dimensiuni `WIDTH x HEIGHT` (7 coloane și 6 rânduri) inițializată cu valoarea `0`, 
care reprezintă o tablă de joc goală.
- `RED` este valoarea care indică că jucătorul care urmează să joace este jucătorul roșu.
Astfel, funcția `init_state()` returnează o stare inițială a jocului, care constă într-o tablă de joc goală și indică că jucătorul roșu va începe primul.
"""

# Funcție ce afișează o stare
def print_state(state):
    for row in range(HEIGHT-1, -1, -1): # Ce face range(HEIGHT-1, -1, -1) ?
        # `range(HEIGHT-1, -1, -1)` generează o secvență de numere care începe de la `HEIGHT-1` (5 în acest caz) și scade până la `-1` (exclusiv), cu un pas de `-1`. 
        # Aceasta înseamnă că bucla va itera de la rândul 5 (ultimul rând) până la rândul 0 (primul rând),
        #  afișând astfel tabla de joc de #!!!!JOS IN SUS, ceea ce este tipic pentru jocurile de tip Connect Four.
        ch = " RA" # `ch` este un șir de caractere care reprezintă simbolurile pentru jucători și celulele goale. Vine de la "ROȘU", "ALBASTRU" și spațiu pentru celule goale.
        #!!!! Dar se poate ch sa primeasca un string in python, ci nu un caracter? Raspuns: Da, în Python, un șir de caractere (string) poate conține unul sau mai multe caractere. 
        # În acest caz, `ch` este un string care conține trei caractere: un spațiu (pentru celulele goale), "R" (pentru jucătorul roșu) și "A" (pentru jucătorul albastru).
        l = map(lambda col: ch[state[BOARD][col][row]], range(WIDTH)) # `l` este o listă care conține simbolurile corespunzătoare pentru fiecare coloană în rândul curent.
         # `map(lambda col: ch[state[BOARD][col][row]], range(WIDTH))` aplică o funcție lambda pentru fiecare coloană din rândul curent,
         #  care ia valoarea din matricea de joc (cine e matrice de joc in codul nostru? Raspuns: Matricea de joc este `state[BOARD]`) 
         # pentru acea coloană și rând și o transformă într-un simbol corespunzător (R pentru roșu, A pentru albastru și spațiu pentru celule goale).
        print("|" + "".join(l) + "|") # Această linie afișează rândul curent al tablei de joc, înconjurat de bare verticale. `join(l)` combină simbolurile din listă într-un singur 
        # șir de caractere.
    print("+" + "".join("-" * WIDTH) + "+") # Această linie afișează o linie de delimitare sub tabla de joc, formată din semnul "+" la început și la sfârșit, și o serie de "-" între ele, 
    # corespunzătoare lățimii tablei.
     # `join("-" * WIDTH)` creează un șir de caractere format din `WIDTH` (7) semne "-" pentru a delimita vizual tabla de joc.
    print("Urmează: %d - %s" % (state[NEXT_PLAYER], name[state[NEXT_PLAYER]]))

#!!!! In python se executa intai print urile si dupa def-urile de la fnctii? Raspuns: În Python, funcțiile trebuie să fie definite înainte de a fi apelate.
# Așadar, dacă încerci să apelezi o funcție înainte de a o defini, vei primi o eroare. În codul de mai sus, funcția `print_state` este definită înainte de a fi apelată,
#  deci nu va exista nicio problemă în acest sens.
# Se afișează starea inițială a jocului
print("Starea inițială:")
print_state(init_state())

Starea inițială:
|       |
|       |
|       |
|       |
|       |
|       |
+-------+
Urmează: 1 - ROȘU


### Mutările

Cum toată informația necesară pentru a descrie o mutare este dată de coloana în care un jucător a ales să își așeze jetonul, o acțiune va fi reprezentată simplu printr-un număr.

**Cerința 1:** Completați funcția `get_available_actions(state)` care primește o stare și întoarce acțiunile corecte (o listă cu acele coloane care nu sunt _pline_).

Funcția `apply_action(state, action)` este deja implementată (întoarce o stare nouă, nu o modifică pe cea dată ca argument).

In [5]:
# Funcție ce întoarce acțiunile valide dintr-o stare dată
def get_available_actions(state):
    # TODO <1>
    # Returnam doar coloanele in care mai exista celule libere
    return [col for col in range(len(state[BOARD])) if 0 in state[BOARD][col]]
    """Linia de cod `return [col for col in range(len(state[BOARD])) if 0 in state[BOARD][col]]` putea fi explicată astfel:
- `range(len(state[BOARD]))` generează o secvență de numere care corespund indexurilor coloanelor din matricea de joc `state[BOARD]`.   
- `col for col in range(len(state[BOARD]))` este o expresie de listă care iterează prin aceste indexuri de coloană și le include în lista rezultată.
- `if 0 in state[BOARD][col]` este o condiție care verifică dacă există cel puțin o celulă liberă (reprezentată de `0`) în coloana curentă `col`. Dacă această condiție este adevărată, 
atunci indexul coloanei `col` este inclus în lista rezultată.

Astfel, această linie de cod returnează o listă de indexuri de coloană care conțin cel puțin o celulă liberă, indicând astfel acțiunile valide pe care un jucător le poate întreprinde 
în starea dată a jocului.


                                                                                                  si rescrisa astfel:
    available_actions = []
    for col in range(len(state[BOARD])):
        if 0 in state[BOARD][col]:
            available_actions.append(col)
    return available_actions

    """

from copy import deepcopy
from functools import reduce

# Funcție ce întoarce starea în care se ajunge prin aplicarea unei acțiuni
def apply_action(state, action):
    if action >= len(state[BOARD]) or 0 not in state[BOARD][action]: # Verificăm dacă acțiunea este validă: dacă indexul coloanei este în afara limitelor sau dacă coloana nu mai 
        # are celule libere (nu conține `0`), atunci acțiunea nu este validă.
        print("Action " + str(action) + " is not valid.")
        return None
    new_board = deepcopy(state[BOARD]) # Creăm o copie adâncă a tablei de joc pentru a nu modifica starea originală. `deepcopy` asigură că toate nivelurile de imbricare ale listei 
    # sunt copiate, astfel încât modificările la `new_board` să nu afecteze `state[BOARD]`.
    new_board[action][new_board[action].index(0,0)] = state[NEXT_PLAYER]
    return (new_board, 3 - state[NEXT_PLAYER])

#!!!! Dar de ce trebuie sa facem o copie? Raspuns: Facem o copie a tablei de joc pentru a păstra starea originală neschimbată. În jocurile de tip Connect Four, 
# fiecare acțiune modifică starea jocului, iar dacă am modifica direct `state[BOARD]`,  am pierde informația despre starea inițială a jocului, ceea ce ar face dificilă gestionarea și 
# urmărirea evoluției jocului.


# Se afișează starea la care se ajunge prin aplicarea unor acțiuni
somestate = reduce(apply_action, [3, 4, 3, 2, 2, 6, 3, 3, 3, 3], init_state())
print_state(somestate)
get_available_actions(somestate) # acțiuni disponibile: [0, 1, 2, 4, 5, 6]

|   A   |
|   R   |
|   A   |
|   R   |
|  RR   |
|  ARA A|
+-------+
Urmează: 1 - ROȘU


[0, 1, 2, 4, 5, 6]

### Stările finale

Pentru a verifica dacă o stare este finală:
 - se verifică dacă ultimul jucător care a mutat a câștigat
 - sau dacă matricea este _plină_
 
Funcția `is_final(state)` va întoarce:
 - `False` dacă starea nu este finală,
 - `1` dacă a câștigat Roșu,
 - `2` dacă a câștigat Albastru și
 - `3` dacă este remiză.

In [6]:
# Funcție ce verifică dacă o stare este finală
def is_final(state) -> bool:
    # Jucătorul care doar ce a mutat ar putea să fie câștigător
    player = 3 - state[NEXT_PLAYER]
    
    ok = lambda pos: all([state[BOARD][c][r] == player for (r, c) in pos]) # `ok` este o funcție lambda care verifică dacă toate pozițiile specificate în `pos` conțin același jucător 
    # (`player`).
     # `all([state[BOARD][c][r] == player for (r, c) in pos])` iterează prin fiecare poziție `(r, c)` din lista `pos` și verifică dacă valoarea din matricea de joc `state[BOARD][c][r]` 
     # este egală cu `player`. Dacă toate pozițiile conțin același jucător, atunci funcția `ok` va returna `True`, indicând că acea configurație de poziții reprezintă o victorie pentru 
     # jucătorul respectiv.
    # Verificăm orizontale
    for row in range(HEIGHT):
        for col in range(WIDTH - 3):
            if ok([(row, col + i) for i in range(4)]): return player
    # Verificăm verticale
    for col in range(WIDTH):
        for row in range(HEIGHT - 3):
            if ok([(row + i, col) for i in range(4)]): return player
    # Verificăm diagonale
    for col in range(WIDTH - 3):
        for row in range(HEIGHT - 3):
            if ok([(row + i, col + i) for i in range(4)]): return player
    for col in range(WIDTH - 3):
        for row in range(HEIGHT - 3):
            if ok([(row + i, col + 3 - i) for i in range(4)]): return player

    # Verificăm dacă matricea este plină
    if not any([0 in col for col in state[BOARD]]): return 3

    return False

In [16]:
# Ajungem la o stare finală oarecare și o afișăm.
from random import choice

rand_state = init_state()
print(rand_state)
while not is_final(rand_state):
    actions = get_available_actions(rand_state)
    if not actions:
        break
    action = choice(get_available_actions(rand_state))
    rand_state = apply_action(rand_state, action)

print_state(rand_state)
print("Învingător: %s" % (name[is_final(rand_state)]))

([[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], 1)
|       |
|     A |
|     R |
|  A  A |
| AR AR |
|AARRRRR|
+-------+
Urmează: 2 - ALBASTRU
Învingător: ROȘU


In [18]:
# Exemplu: Se afișează starea obținută prin aplicarea unor acțiuni
all_actions = [1, 2, 1, 3, 1, 4, 2, 5]
some_state = reduce(apply_action, all_actions, init_state())
print_state(some_state)
print("Învingător: %s" % (name[is_final(some_state)]))

|       |
|       |
|       |
| R     |
| RR    |
| RAAAA |
+-------+
Urmează: 1 - ROȘU
Învingător: ALBASTRU


## Algoritmul UCT

Algoritmii din familia MCTS conțin patru etape importante:
 - **selecție** - o strategie de alegere a unei acțiuni pentru a exploata
 - **extindere** - construirea unui nod nou în arbore
 - **simulare** - desfășurarea unui joc în mod aleator către o stare finală
 - **propagare înapoi** - actualizarea statisticilor pentru toate nodurile


### Reprezentarea unui nod

Un nod din arborele de stări va fi un dicționar ce conține:
 - numărul de vizitări `N` -- de câte ori s-au realizat simulări din acel nod sau dintr-un descendent al său.
 - valoarea estimată `Q` -- o indicație a calității nodului, bazat pe numărul jocurilor câștigate pornind din acel nod.
 - o referință la starea care îi corespunde -- `STATE`
 - o referință către nodul părinte -- `PARENT`
 - lista copiilor -- `ACTIONS` -- un dicționar care conține, pentru fiecare acțiune explorată, o referință către nodul următor

Exemplu de nod corespunzător unei stări.

    {N: 7, Q: 2.5, STATE: ([[0, 0, 0, 0, 0, 0], ...], 1), PARENT: None, ACTIONS: {0: {N: 3, ...}, 1: {N: 4, ...}}}
    
### Desfășurarea algoritmului

1. Dacă algoritmul pornește cu un arbore gol (fără memorie), atunci se construiește un nod nou.
   Altfel se alege subarborele corespunzător ultimei acțiuni a adversarului. (`TODO3`)

2. Până când se atinge limita bugetului de calcul:
  1. pornind din rădăcină, se alege succesiv un nod următor până când se atinge o stare finală (`is_final`) sau un nod din care nu s-au explorat toate acțiunile posibile (`TODO2` și `TODO4`)
  2. pentru un nod care nu este final și din care nu s-au explorat toate acțiunile, se construiește un nod-copil pentru una dintre acțiunile neexplorate
  3. se simulează un joc pornind din nodul nou până într-o stare finală (`TODO5`)
  4. se evaluează starea finală și se calculează o recompensă (`TODO6`)
  5. se propagă înapoi acea recompensă, actualizându-se și statisticile (numărul de vizite) pentru fiecare nod până la rădăcină (`TODO7`)

In [19]:
# Constante

N = 'N'
Q = 'Q'
STATE = 'state'
PARENT = 'parent'
ACTIONS = 'actions'

### Afișarea unui arbore

In [20]:
def print_tree(tree, indent = 0, maxdepth = None):
    if not tree:
        return
    advance = maxdepth is None or maxdepth > 0 # Verificăm dacă trebuie să continuăm afișarea subarborilor. 
    #Dacă `maxdepth` este `None`, înseamnă că nu există o limită de adâncime și vom continua să afișăm subarborii. Dacă `maxdepth` este un număr, atunci vom continua să afișăm 
    # subarborii doar dacă `maxdepth` este mai mare decât 0.
    tab = "".join(" " * indent)
    print(f"{tab}N = {tree[N]}, Q = {float(tree[Q]):.4} {'' if advance else ', A = [' + ', '.join([str(a) + ':...' for a in tree[ACTIONS]]) + ']'}")
    if advance:
        for a in tree[ACTIONS]:
            print(f"{tab} {a} ==> ")
            print_tree(tree[ACTIONS][a], indent + 3, maxdepth if maxdepth is None else maxdepth - 1)

### Algoritm

In [21]:
# Funcție ce întoarce un nod nou,
# eventual copilul unui nod dat ca argument
def init_node(state, parent = None):
    return {N: 0, Q: 0, STATE: state, PARENT: parent, ACTIONS: {}}

Selecția se face pe un node care are toate acțiunile explorate, și se alege acțiunea $a$ care maximizează

$ \frac{Q_a}{N_a} + c \cdot \sqrt{\frac{2 \cdot \log{N_{node}}}{N_a}} $

unde $N_a$ și $Q_a$ corespund copilului corespunzător acțiunii $a$, iar $N_{node}$ corespunde nodului pentru care facem selecția (parametrul `node`).

In [22]:
from math import sqrt, log
from platform import node

# constanta care reglează raportul între explorare și exploatare (CP = 0 -> doar exploatare)
CP = 1.0 / sqrt(2.0)

# Funcție ce alege o acțiune dintr-un nod
def select_action(node, c = CP):
    N_node = node[N]
    # TODO <2>
    # Se caută acțiunea a care maximizează expresia:
    # Q_a / N_a  +  c * sqrt(2 * log(N_node) / N_a)
    # Calculam scorul UCT pentru fiecare actiune si alegem maximul
    def uct_score(action):
        child = node[ACTIONS][action]
        N_a = child[N] # Numărul de vizite pentru acțiunea `action` (sau nodul copil corespunzător acestei acțiuni).
         # `child[N]` accesează valoarea `N` din nodul copil asociat acțiunii `action`, care reprezintă numărul de vizite (vezi mai sus)
        Q_a = child[Q] # Valoarea totală a recompenselor acumulate pentru acțiunea `action` (sau nodul copil corespunzător acestei acțiuni).
         # `child[Q]` accesează valoarea `Q` din nodul copil asociat acțiunii `action`, care reprezintă suma recompenselor acumulate pentru acea acțiune.
         # Aceasta este o măsură a performanței acțiunii respective în simulările anterioare.
        return Q_a / N_a + c * sqrt(2 * log(N_node) / N_a)
    return max(node[ACTIONS].keys(), key=uct_score)

# Scurtă testare; stările nu sunt relevante aici
test_root = {N: 6, Q: 0.75, STATE: None, PARENT: None, ACTIONS: {}}
test_root[ACTIONS][3] = {N: 4, Q: 0.9, STATE: None, PARENT: test_root, ACTIONS: {}}
test_root[ACTIONS][5] = {N: 2, Q: 0.1, STATE: None, PARENT: test_root, ACTIONS: {}}

print(select_action(test_root, CP))  # ==> 5 (0.8942 < 0.9965)
print(select_action(test_root, 0.3)) # ==> 3 (0.50895 > 0.45157)

5
3


In [23]:
# Algoritmul MCTS (UCT)
#  state0 - starea pentru care trebuie aleasă o acțiune
#  budget - numărul de iterații permis / numărul de noduri care va fi construit
#  tree - un arbore din explorările anterioare, dacă există
#  opponent_s_action - ultima acțiune a adversarului, dacă există

def mcts(state0, budget, tree, opponent_s_action = None):
    # TODO <3>
    # DACĂ există un arbore construit anterior ȘI
    #   acesta are un copil ce corespunde ultimei acțiuni a adversarului,
    # ATUNCI acel copil va deveni nodul de început pentru algoritm.
    # ALTFEL, arborele de start este un nod gol.

    if tree and opponent_s_action is not None and opponent_s_action in tree[ACTIONS]:
        tree = tree[ACTIONS][opponent_s_action]
    else:
        tree = init_node(state0)
    """Ce face if ul de mai sus?
    - `if tree and opponent_s_action is not None and opponent_s_action in tree[ACTIONS]` verifică dacă există un arbore construit anterior (`tree`), dacă ultima acțiune a adversarului 
    (`opponent_s_action`) nu este `None`, adică există o acțiune a adversarului, și dacă această acțiune există ca un copil în arborele curent (`tree[ACTIONS]`).
    - Dacă toate aceste condiții sunt adevărate, atunci arborele de start pentru algoritm va fi copilul corespunzător ultimei acțiuni a adversarului (`tree[ACTIONS][opponent_s_action]`).
    - Dacă oricare dintre aceste condiții nu este adevărată, atunci arborele de start va fi un nod nou, inițializat cu starea `state0` și fără părinte (`init_node(state0)`).
    Această logică permite algoritmului MCTS să continue explorarea de la starea curentă a jocului, în loc să înceapă de la starea inițială, dacă există deja un arbore construit care 
    corespunde ultimei acțiuni a adversarului.
    """
    
    #---------------------------------------------------------------

    for x in range(budget):
        # Pornim procesul de selecție din nodul rădăcină / starea inițială
        node = tree

        # TODO <4>
        # Mergem din părinte în copil până când ajungem într-un nod care
        # - corespunde unei stări finale, sau
        # - nu are toți copiii construiți (mai sunt acțiuni disponibile 
        #   pentru care nu au fost construite noduri copil)
        # Dacă nu am ajuns într-un astfel de nod, mergem în copilul selectat
        # folosind select_action().

        while True:
            if is_final(node[STATE]):
                break
            actions = get_available_actions(node[STATE])
            if len(node[ACTIONS]) < len(actions):
                break
            action = select_action(node)
            node = node[ACTIONS][action]

        """Ce face while ul de mai sus?
        - `while True:` inițiază un loop infinit care va continua până când se întâlnește o condiție de oprire în interiorul loop-ului.
        - `if is_final(node[STATE]): break` verifică dacă starea asociată nodului curent (`node[STATE]`) este o stare finală a jocului. Dacă este adevărat, atunci se iese din loop
          folosind `break`.
        - `actions = get_available_actions(node[STATE])` obține lista de acțiuni disponibile pentru starea curentă a nodului.
        - `if len(node[ACTIONS]) < len(actions): break` verifică dacă numărul de copii construiți pentru nodul curent (`len(node[ACTIONS])`) este mai mic decât numărul de acțiuni 
        disponibile (`len(actions)`). Dacă este adevărat, înseamnă că există acțiuni pentru care nu au fost construite noduri copil, deci se iese din loop folosind `break`.

        Si cand se apeleaza ce de-al doilea if? 
        Raspuns: Al doilea `if` se apelează după ce s-a verificat dacă starea curentă este finală. Dacă starea nu este finală, atunci se obțin acțiunile disponibile pentru acea stare.

        #!!!! Pai daca primul if si cel de-al doilea if sunt in while ul infinit, nu crezi ca cel de-al doilea if va scoate programul mai repede decat cand trebuia sa se scoata
        #!!!! la momemntul potrivit? 
        # 
        # Raspuns: Da, al doilea `if` poate scoate programul mai repede decât primul `if` dacă există acțiuni disponibile pentru care nu au fost construite noduri copil.
        # 
        # Dar asta nu e o problema, pentru ca in momentul in care se intalneste un nod care nu are toti copiii construiti, se va iesi din loop si se va trece la etapa urmatoare a 
        # algoritmului MCTS, care este construirea unui nod nou pentru o acțiune neexplorată și apoi simularea unei desfășurări a jocului. Deci, chiar dacă al doilea `if` poate scoate 
        # programul mai repede, acest lucru este în concordanță cu logica algoritmului și nu reprezintă o problemă.
        #
        # Dar daca primul if se intalneste mai repede, nu e o problema? Raspuns: Nu, dacă primul `if` se întâlnește mai repede, nu este o problemă, deoarece acest lucru indică că am ajuns
        #  într-o stare finală a jocului, ceea ce este un punct de oprire valid pentru procesul de selecție. În acest caz, nu mai este necesar să verificăm dacă există acțiuni neexplorate,
        #  deoarece starea finală nu va avea acțiuni disponibile. Astfel, indiferent dacă primul `if` sau al doilea `if` se întâlnește mai repede, ambele scenarii sunt în 
        # concordanță cu logica algoritmului MCTS și nu reprezintă o problemă.

        #!!!! Deci, în concluzie, ambele `if`-uri sunt necesare pentru a asigura că procesul de selecție se oprește în mod corespunzător atunci când se întâlnește o stare finală sau un 
        #!!!! nod care nu are toți copiii construiți, și ambele scenarii sunt valide în contextul algoritmului MCTS.
        

        - `action = select_action(node)` selectează o acțiune folosind funcția `select_action`, care aplică strategia UCT pentru a alege cea mai promițătoare acțiune dintre cele 
        disponibile.
        - `node = node[ACTIONS][action]` actualizează nodul curent pentru a fi copilul corespunzător acțiunii selectate, continuând astfel procesul de selecție în subarborele corespunzător
          acelei acțiuni.
        Acest loop asigură că procesul de selecție continuă să coboare în arborele de căutare până când se întâlnește un nod care este fie o stare finală, fie un nod care nu are toți 
        copiii construiți, moment în care se va opri și se va trece la următoarea etapă a algoritmului MCTS.            
        """
        
        #---------------------------------------------------------------
        
        # TODO <5>
        # Dacă am ajuns într-un nod care nu este final și care nu are
        # copii corespunzători tuturor acțiunilor disponibile,
        # construim un nod nou, corespunzător unei acțiuni neexplorate încă.

        if not is_final(node[STATE]): # Verificăm dacă nodul curent nu este o stare finală. Dacă nu este final, atunci putem continua să construim un nod nou pentru o acțiune neexplorată.
            actions = get_available_actions(node[STATE]) # Obținem lista de acțiuni disponibile pentru starea curentă a nodului.
             # `get_available_actions(node[STATE])` returnează o listă de acțiuni valide pentru starea asociată nodului curent. Aceste acțiuni reprezintă opțiunile pe care un jucător
             #  le poate alege
             #  în acea stare a jocului
            untried = [a for a in actions if a not in node[ACTIONS]] # Creăm o listă `untried` care conține acțiunile disponibile pentru care nu au fost construite noduri copil în
            # arborele curent.
             # `untried = [a for a in actions if a not in node[ACTIONS]]` iterează prin fiecare acțiune `a` din lista de acțiuni disponibile și include în lista `untried` doar acele 
             # acțiuni care nu există ca
             #  chei în dicționarul `node[ACTIONS]`, adică acele acțiuni pentru care nu au fost construite noduri copil în arborele curent.

             # Astfel, `untried` va conține doar acțiunile care sunt disponibile în starea curentă, dar pentru care nu există încă noduri copil în arborele de căutare, 
             # indicând astfel acțiunile neexplorate.
            if untried:
                action = choice(untried) # Selectăm aleatoriu o acțiune din lista `untried`, care conține acțiunile disponibile pentru care nu au fost construite noduri copil.
                new_state = apply_action(node[STATE], action) # Aplicăm acțiunea selectată asupra stării asociate nodului curent pentru a obține o nouă stare a jocului.
                node[ACTIONS][action] = init_node(new_state, node) # Construim un nod nou pentru acțiunea selectată și starea rezultată, și îl 
                # adăugăm ca un copil al nodului curent în arborele de căutare. 
                node = node[ACTIONS][action] # Actualizăm nodul curent pentru a fi nodul nou construit, astfel încât să continuăm procesul de simulare de la această nouă stare.
        
        #---------------------------------------------------------------
        
        # TODO <6>
        # Din nodul nou construit, se simulează o desfășurare a jocului până
        # la ajungerea într-o starea finală. Se evaluează recompensa în acea stare.
        state = node[STATE]
        while not is_final(state):
            actions = get_available_actions(state)
            if not actions: # Dacă nu există acțiuni disponibile, înseamnă că am ajuns într-o stare finală (deși acest lucru ar trebui să fie deja verificat de `is_final(state)`)
                break
            action = choice(actions)
            state = apply_action(state, action) # Aplicăm acțiunea selectată asupra stării curente pentru a obține o nouă stare a jocului, 
            # continuând astfel simularea până când se ajunge într-o stare finală.



        while not is_final(state):
            break
        
        winner = is_final(state)
        if winner == state0[NEXT_PLAYER]:
            reward = 1
        elif winner == (3 - state0[NEXT_PLAYER]):
            reward = 0.0
        elif winner == 3:
            reward = 0.25
        else:
            reward = 0.5
        #---------------------------------------------------------------

        # TODO <7>
        # Se actualizează toate nodurile de la nodul curent către rădăcină:
        #  - se incrementează valoarea N din fiecare nod;
        #  - pentru nodurile unde tocmai a mutat acest jucător (urmează celălalt jucător),
        #    se adună recompensa la valoarea Q;
        #  - pentru nodurile celelalte, valoarea Q se adună 1 cu minus recompensa.
        
        while node is not None: # Continuăm să actualizăm nodurile până când ajungem la rădăcină (unde `node[PARENT]` va fi `None`).
            node[N] += 1 # Incrementăm valoarea `N` din nodul curent pentru a reflecta faptul că acest nod a fost vizitat încă o dată în procesul de selecție și simulare.
            if node[STATE][NEXT_PLAYER] != state0[NEXT_PLAYER]: # Verificăm dacă jucătorul care urmează să joace în starea asociată nodului curent este diferit de jucătorul care a 
                # început procesul de selecție (jucătorul pentru care se alege o acțiune).
                node[Q] += reward # Dacă jucătorul care urmează să joace în starea nodului curent este diferit de jucătorul pentru care se alege o acțiune, atunci adăugăm recompensa 
                # la valoarea `Q` a nodului, (Q  o valoare alocată nodului care reprezintă suma recompenselor acumulate pentru acea stare în simulările anterioare)
                #  deoarece acest nod reprezintă o situație în care adversarul a mutat
            else:
                node[Q] += 1 - reward # Dacă jucătorul care urmează să joace în starea nodului curent este același cu jucătorul pentru care se alege o acțiune, atunci adăugăm `1 - reward`
                # la valoarea `Q` a nodului,
                #  deoarece acest nod reprezintă o situație în care jucătorul a mutat și vrem să reflectăm faptul că o recompensă mai mare pentru adversar este o penalizare pentru 
                # jucătorul curent.   

                """Nu prea inteleg de ce adaugam 1 - reward in acest caz, nu ar trebui sa adaugam reward? 
                
                Raspuns: În algoritmul MCTS, atunci când actualizăm nodurile după o simulare, vrem să reflectăm faptul că o recompensă mai mare pentru adversar este o penalizare pentru 
                jucătorul curent.
                - Dacă nodul curent reprezintă o situație în care adversarul a mutat (adică `node[STATE][NEXT_PLAYER] != state0[NEXT_PLAYER]`), atunci adăugăm recompensa direct la `Q` 
                pentru a reflecta performanța adversarului în acea stare.
                - Dacă nodul curent reprezintă o situație în care jucătorul curent a mutat (adică `node[STATE][NEXT_PLAYER] == state0[NEXT_PLAYER]`), 
                atunci vrem să penalizăm această stare dacă adversarul a obținut o recompensă mare în simulare.
                De aceea, adăugăm `1 - reward` la `Q` în acest caz, pentru a reflecta faptul că o recompensă mai mare pentru adversar (un `reward` mai mare) 
                va duce la o penalizare mai mare pentru jucătorul curent (un `1 - reward` mai mic). Astfel, această logică ajută la echilibrarea valorilor `Q` în funcție de performanța 
                atât a jucătorului curent, cât și a adversarului în simulările anterioare."""
            node = node[PARENT] # Actualizăm nodul curent pentru a fi părintele său, continuând astfel să urcăm în arborele de căutare până când ajungem la rădăcină (unde `node[PARENT]` va fi `None` și loop-ul se va opri).  

        #---------------------------------------------------------------

    if tree:
        final_action = select_action(tree, 0.0)
        return (final_action, tree[ACTIONS][final_action])
    # Acest cod este aici doar ca să nu dea erori testele mai jos; în mod normal tree nu trebuie să fie None
    if get_available_actions(state0):
        return (get_available_actions(state0)[0], init_node(state0))
    return (0, init_node(state0))

In [33]:
# Testare MCTS
(action, tree) = mcts(init_state(), 20, None, None)
print("Acțiunea aleasă: ", action)
if tree: print_tree(tree[PARENT]) # Trebuie ca arborele să fie destul de echilibrat, nu dezechilibrat

Acțiunea aleasă:  0
N = 20, Q = 14.0 
 4 ==> 
   N = 2, Q = 0.0 
    2 ==> 
      N = 1, Q = 1.0 
 5 ==> 
   N = 2, Q = 0.0 
    5 ==> 
      N = 1, Q = 1.0 
 3 ==> 
   N = 3, Q = 1.0 
    3 ==> 
      N = 1, Q = 1.0 
    0 ==> 
      N = 1, Q = 1.0 
 0 ==> 
   N = 5, Q = 3.0 
    5 ==> 
      N = 1, Q = 0.0 
    3 ==> 
      N = 1, Q = 1.0 
    0 ==> 
      N = 1, Q = 0.0 
    6 ==> 
      N = 1, Q = 1.0 
 6 ==> 
   N = 2, Q = 0.0 
    4 ==> 
      N = 1, Q = 1.0 
 1 ==> 
   N = 4, Q = 2.0 
    3 ==> 
      N = 1, Q = 0.0 
    0 ==> 
      N = 1, Q = 1.0 
    4 ==> 
      N = 1, Q = 1.0 
 2 ==> 
   N = 2, Q = 0.0 
    3 ==> 
      N = 1, Q = 1.0 


## Evaluarea algoritmului

Funcția de mai jos opune doi jucători ce folosesc algoritmul UCT pentru a decide asupra acțiunii dintr-o stare.

In [34]:
def play_games(games_no, budget1, budget2, verbose = False):
    # Efortul de căutare al jucătorilor
    budget = [budget1, budget2]
    
    score = {p: 0 for p in name}
        
    for i in range(games_no):
        # Memoriile inițiale
        memory = [None, None]
        
        # Se desfășoară jocul
        state = init_state()
        last_action = None
    
        while state and not is_final(state):
            p = state[NEXT_PLAYER] - 1
            (action, memory[p]) = mcts(state, budget[p], memory[p], last_action)
            state = apply_action(state, action)
            last_action = action
        
        # Cine a câștigat?
        if(state):
            winner = is_final(state)
            score[name[winner]] += + 1
        
        # Afișăm
        if verbose and state:
            print_state(state)
            if winner == 3: print("Remiză.")
            else: print("A câștigat %s" % name[winner])

    # Afișează scorul final
    if verbose:
        print("Scor final: %s." % (str(score)))
    return score

### Testare

Rulați cele 2 serii de câte 5 jocuri, și verificați că la sfârșit câștigă jucătorul cu buget mai mare.

Apoi, dați o valoare (e.g. 20) pentru `ngames` și observați cum se modifică rezultatele în funcție de distribuția bugetului.

In [42]:
# play_games(N, BR, BA, VERBOSE) - rulează N jocuri, cu bugetele BR pt ROȘU și BA pt ALBASTRU
# TODO: rulați pentru câte 5 jocuri:
play_games(5, 2, 30, True) # ne așteptăm să câștige ALBASTRU
play_games(5, 30, 2, True) # ne așteptăm să câștige ROȘU

# TODO: rulați pentru câte 20 de jocuri:
ngames = 20
print(f"Rezultate pentru câte {ngames} de jocuri (ROȘU / ALBASTRU / REMIZĂ):")
print("Buget mic | Buget mare | avantaj ALBASTRU | avantaj ROȘU")
for small, big in [(2, 30), (5, 30), (10, 30), (20, 20)]:
    print(f"{small : >5}     | {big : >6}     |", end = "")
    score = play_games(ngames, small, big, False)
    print("{:>16}".format(f"{score[name[RED]]} / {score[name[BLUE]]} / {score[name[DRAW]]}"), end = "  |")
    score = play_games(ngames, big, small, False)
    print("{:>12}".format(f"{score[name[RED]]} / {score[name[BLUE]]} / {score[name[DRAW]]}"), end = "  |")
    print()
    

|       |
|       |
|  R    |
|  R  A |
| AR AR |
| ARRRAA|
+-------+
Urmează: 2 - ALBASTRU
A câștigat ROȘU
|       |
|       |
|  RAA  |
|R AAR  |
|R RAR  |
|A RAAR |
+-------+
Urmează: 1 - ROȘU
A câștigat ALBASTRU
|       |
|       |
| A    R|
| R A  R|
| A A AR|
| RARARR|
+-------+
Urmează: 2 - ALBASTRU
A câștigat ROȘU
|       |
|       |
|       |
|       |
|R  AR R|
|RAAAA R|
+-------+
Urmează: 1 - ROȘU
A câștigat ALBASTRU
|       |
|       |
|       |
|      R|
|AAAA  R|
|RRAR AR|
+-------+
Urmează: 1 - ROȘU
A câștigat ALBASTRU
Scor final: {'': 0, 'ROȘU': 2, 'ALBASTRU': 3, 'REMIZĂ': 0}.
|       |
|       |
|       |
|    A  |
|R A R  |
|AARRRRA|
+-------+
Urmează: 2 - ALBASTRU
A câștigat ROȘU
|       |
|       |
|       |
|      A|
|   RRRR|
| AARARA|
+-------+
Urmează: 2 - ALBASTRU
A câștigat ROȘU
|       |
|    R  |
|    R  |
|  A R  |
|  A RA |
|RAR AAR|
+-------+
Urmează: 2 - ALBASTRU
A câștigat ROȘU
|       |
|       |
|       |
|       |
|    A  |
|A ARRRR|
+-------+
Urmeaz

In [43]:
20

20